The 1 Production Rule
> **Assume upstream data will break — eventually.**

Design your pipeline so that:
- failures are loud
- damage is contained
- rollback is possible

### Pandas Failure Modes (VERY COMMON)
1. Memory Explosion
```python
pd.read_csv("huge_file.csv")
```
- Killed by OOM
- No warning
- No logs

2. Schema Drift
```python
df["salary"]   # column renamed upstream
```
- KeyError
- Or worse: silent NaNs

3. Row Explosion
```python
users.merge(events)
```
- 1M rows -> 200M rows
- Pipeline “runs” but is wrong

4. Time Leakage
```python
rolling("30D")
```
- Works fine
- Model accuracy fake
- Production failure

### Memory-Safe Pandas Patterns
#### Always control dtypes
```python
dtypes = {
    "user_id": "int32",
    "amount": "float32",
    "city": "category"
}

df = pd.read_csv("data.csv", dtype=dtypes)
```

#### Chunked reading
```python
chunks = pd.read_csv("data.csv", chunksize=100_000)

for chunk in chunks:
    process(chunk)
```

### Idempotent Pipelines
Production pipelines **must be rerunnable.**

#### Bad
```python
df["count"] += 1
```

#### Good
```python
df["count"] = compute_count(df)
```

- Same input -> same output
- No hidden state

### Late & Partial Data
**Scenario**
- Events arrive hours late
- Yesterday’s data changes

**Strategy**
- recompute rolling windows
- use watermarks
- never “append-only” blindly

### Logging & Observability
At minimum, log:
```text
rows_in
rows_out
null_rate_per_column
min/max timestamps
```

Example:
```python
print(f"Rows in: {len(df)} | Rows out: {len(out)}")
```

### Starting Point (INTENTIONALLY FRAGILE PIPELINE)

You are given a **pipeline that works today**, but is **unsafe in production.**

In [5]:
import pandas as pd

def build_features(users, events):
    df = users.merge(events, on="user_id", how="left")

    df["event_time"] = pd.to_datetime(df["event_time"])
    df["amount"] = df["amount"].fillna(0)

    features = (
        df
        .groupby("user_id")
        .agg(
            total_amount=("amount", "sum"),
            event_count=("event_id", "count"),
            last_event_time=("event_time", "max")
        )
        .reset_index()
    )

    return features


### Exercise 1
Write a function:
```python
def validate_input_schema(users, events):
    ...
```

Rules:
- Required columns:
    - users: user_id
    - events: event_id, user_id, event_time, amount
- No extra columns allowed
- Raise immediately if violated

In [6]:
def validate_input_schema(users, events):
    required_users = {"user_id"}
    required_events = {"event_id", "user_id", "event_time", "amount"}

    assert set(users.columns) == required_users, "Users schema mismatch"
    assert set(events.columns) == required_events, "Events schema mismatch"

In [7]:
def optimize_dtypes(events):
    events = events.copy()
    events["user_id"] = events["user_id"].astype("int32")
    events["amount"] = events["amount"].astype("float32")
    return events

### Exercise 2 
Replace this:
```python
users.merge(events, on="user_id", how="left")
```
With:
- explicit `validate=`
- explanation of **why this matters**

In [8]:
def safe_merge(users, events):
    return users.merge(
        events,
        on="user_id",
        how="left",
        validate="one_to_many"
    )

### Exercise 3
Add:
- a cutoff_time parameter
- filtering logic so:
```python
event_time < cutoff_time
```

Explain:
- why `<= cutoff_time` is unsafe
- why this belongs **before aggregation**

In [9]:
def filter_by_cutoff(events, cutoff_time):
    events = events.copy()
    events["event_time"] = pd.to_datetime(events["event_time"])
    return events[events["event_time"] < cutoff_time]


### Exercise 4
Refactor:
- dtypes for `amount`
- dtypes for `user_id`

In [10]:
def optimize_dtypes(events):
    events = events.copy()
    events["user_id"] = events["user_id"].astype("int32")
    events["amount"] = events["amount"].astype("float32")
    return events


### Exercise 5
Add logging/metrics:
- rows before merge
- rows after merge
- rows after aggregation
- % missing per column

In [11]:
def log_metrics(stage, df):
    print(f"[{stage}] rows = {len(df)}")
    print(df.isna().mean())

### Exercise 6
Write:
```python
def validate_output(features):
    ...
```
Assert:
- one row per `user_id`
- no negative amounts
- `event_count >= 0`
- `last_event_time` ≤ cutoff_time

In [12]:
def validate_output(features, cutoff_time):
    assert features["user_id"].is_unique
    assert (features["total_amount"] >= 0).all()
    assert (features["event_count"] >= 0).all()
    assert features["last_event_time"].max() <= cutoff_time

In [13]:
def build_features_production(users, events, cutoff_time):
    validate_input_schema(users, events)

    events = optimize_dtypes(events)

    events = filter_by_cutoff(events, cutoff_time)
    log_metrics("after_cutoff", events)

    merged = safe_merge(users, events)
    log_metrics("after_merge", merged)

    features = (
        merged
        .groupby("user_id", as_index=False)
        .agg(
            total_amount=("amount", "sum"),
            event_count=("event_id", "count"),
            last_event_time=("event_time", "max")
        )
    )
    log_metrics("after_aggregation", features)

    validate_output(features, cutoff_time)

    return features
